# SAC

In [ ]:
# Minimal SAC (continuous) — PyTorch + Gymnasium
# ----------------------------------------------------------
# - Twin Q networks with target networks
# - Gaussian policy with Tanh squashing
# - Automatic entropy tuning (temperature alpha)
# - Replay buffer, Polyak target updates
# ----------------------------------------------------------

In [ ]:


from __future__ import annotations
import random
from dataclasses import dataclass
from collections import deque, namedtuple
from typing import Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym

# ----------------------------
# Hyperparameters
# ----------------------------
ENV_ID            = "Pendulum-v1"    # continuous action env
TOTAL_STEPS       = 200_000
WARMUP_STEPS      = 1_000            # act randomly before SAC updates
UPDATE_EVERY      = 1                 # learn every K env steps
GRAD_STEPS        = 1                 # gradient steps per update
BUFFER_CAPACITY   = 1_000_000
BATCH_SIZE        = 256
GAMMA             = 0.99
TAU               = 0.005            # Polyak factor for target nets
LR_ACTOR          = 3e-4
LR_CRITIC         = 3e-4
LR_ALPHA          = 3e-4             # entropy temperature optimizer
SEED              = 42
DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# Utilities
# ----------------------------
def set_seed_everywhere(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def soft_update_(target: nn.Module, online: nn.Module, tau: float = TAU):
    with torch.no_grad():
        for tp, op in zip(target.parameters(), online.parameters()):
            tp.data.mul_(1.0 - tau).add_(tau * op.data)

Transition = namedtuple("Transition", ("state", "action", "next_state", "reward", "done"))

class ReplayBuffer:
    def __init__(self, capacity=BUFFER_CAPACITY):
        self.buf = deque(maxlen=capacity)
    def push(self, *args):
        self.buf.append(Transition(*args))
    def __len__(self): return len(self.buf)
    def sample(self, batch_size: int) -> Transition:
        batch = random.sample(self.buf, batch_size)
        return Transition(*zip(*batch))

# ----------------------------
# Networks
# ----------------------------
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x): return self.net(x)

class QCritic(nn.Module):
    """Twin Q networks: Q1(s,a), Q2(s,a)"""
    def __init__(self, obs_dim, act_dim, hidden=256):
        super().__init__()
        self.q1 = MLP(obs_dim + act_dim, 1, hidden)
        self.q2 = MLP(obs_dim + act_dim, 1, hidden)
    def forward(self, s, a):
        x = torch.cat([s, a], dim=-1)
        return self.q1(x), self.q2(x)

class GaussianPolicy(nn.Module):
    """Gaussian policy with Tanh squashing and log-prob correction."""
    def __init__(self, obs_dim, act_dim, act_low, act_high, hidden=256):
        super().__init__()
        self.mu = MLP(obs_dim, act_dim, hidden)
        self.log_std = MLP(obs_dim, act_dim, hidden)
        # action scaling to env bounds
        self.register_buffer("act_low",  torch.as_tensor(act_low,  dtype=torch.float32))
        self.register_buffer("act_high", torch.as_tensor(act_high, dtype=torch.float32))
        self.register_buffer("act_scale", (self.act_high - self.act_low)/2.0)
        self.register_buffer("act_bias",  (self.act_high + self.act_low)/2.0)

    def forward(self, s: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        mu = self.mu(s)
        log_std = self.log_std(s).clamp(-20, 2)   # stabilize std
        return mu, log_std

    @staticmethod
    def _tanh_correction_logprob(u: torch.Tensor) -> torch.Tensor:
        # log |det d(tanh(u))/du| = sum log(1 - tanh(u)^2)
        return torch.sum(torch.log(1 - torch.tanh(u) ** 2 + 1e-6), dim=-1)

    def sample(self, s: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Sample action (reparam) and log_prob under tanh-squashed Gaussian. Action is scaled to env range."""
        mu, log_std = self.forward(s)
        std = log_std.exp()
        dist = torch.distributions.Normal(mu, std)
        # Reparameterization trick
        u = dist.rsample()                     # pre-squash action
        a = torch.tanh(u)                      # squash
        # Log prob with tanh correction (before scaling)
        log_prob = torch.sum(dist.log_prob(u), dim=-1) - self._tanh_correction_logprob(u)
        # Scale to env bounds (affine). Constant log|det| from scale omitted (doesn't affect gradients wrt θ).
        a_scaled = a * self.act_scale + self.act_bias
        return a_scaled, log_prob

    def act_deterministic(self, s: torch.Tensor) -> torch.Tensor:
        mu, _ = self.forward(s)
        a = torch.tanh(mu)
        return a * self.act_scale + self.act_bias

# ----------------------------
# SAC Agent (SAC v2: no V-net)
# ----------------------------
@dataclass
class SACConfig:
    gamma: float = GAMMA
    tau: float = TAU
    batch_size: int = BATCH_SIZE
    lr_actor: float = LR_ACTOR
    lr_critic: float = LR_CRITIC
    lr_alpha: float = LR_ALPHA

class SACAgent:
    def __init__(self, env: gym.Env, cfg: SACConfig = SACConfig()):
        self.env = env
        obs_dim = env.observation_space.shape[0]
        act_dim = env.action_space.shape[0]
        act_low  = env.action_space.low
        act_high = env.action_space.high

        self.actor  = GaussianPolicy(obs_dim, act_dim, act_low, act_high).to(DEVICE)
        self.q_net  = QCritic(obs_dim, act_dim).to(DEVICE)
        self.q_tgt  = QCritic(obs_dim, act_dim).to(DEVICE)
        self.q_tgt.load_state_dict(self.q_net.state_dict())

        self.actor_opt = optim.Adam(self.actor.parameters(), lr=cfg.lr_actor)
        self.q_opt     = optim.Adam(self.q_net.parameters(),   lr=cfg.lr_critic)

        # Automatic entropy tuning
        self.target_entropy = -float(act_dim)     # heuristic: -|A|
        self.log_alpha = torch.zeros(1, device=DEVICE, requires_grad=True)
        self.alpha_opt = optim.Adam([self.log_alpha], lr=cfg.lr_alpha)

        self.cfg = cfg
        self.replay = ReplayBuffer()

    @property
    def alpha(self) -> torch.Tensor:
        return self.log_alpha.exp()

    def update(self):
        """One SAC update step (critic -> actor -> alpha)."""
        if len(self.replay) < self.cfg.batch_size:
            return None

        batch = self.replay.sample(self.cfg.batch_size)
        s = torch.as_tensor(np.stack(batch.state),      dtype=torch.float32, device=DEVICE)
        a = torch.as_tensor(np.stack(batch.action),     dtype=torch.float32, device=DEVICE)
        r = torch.as_tensor(np.stack(batch.reward),     dtype=torch.float32, device=DEVICE).unsqueeze(-1)  # [B,1]
        d = torch.as_tensor(np.stack(batch.done),       dtype=torch.float32, device=DEVICE).unsqueeze(-1)  # [B,1]
        ns= torch.as_tensor(np.stack(batch.next_state), dtype=torch.float32, device=DEVICE)

        # --- Critic update ---
        with torch.no_grad():
            a2, logp2 = self.actor.sample(ns)                # next action and logπ
            q1_t, q2_t = self.q_tgt(ns, a2)
            q_t_min = torch.minimum(q1_t, q2_t)
            y = r + self.cfg.gamma * (1.0 - d) * (q_t_min - self.alpha * logp2.unsqueeze(-1))  # [B,1]

        q1, q2 = self.q_net(s, a)
        critic_loss = nn.functional.mse_loss(q1, y) + nn.functional.mse_loss(q2, y)

        self.q_opt.zero_grad(set_to_none=True)
        critic_loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), max_norm=1.0)
        self.q_opt.step()

        # --- Actor update ---
        a_pi, logp = self.actor.sample(s)
        q1_pi, q2_pi = self.q_net(s, a_pi)
        q_pi_min = torch.minimum(q1_pi, q2_pi)
        actor_loss = (self.alpha.detach() * logp - q_pi_min.squeeze(-1)).mean()

        self.actor_opt.zero_grad(set_to_none=True)
        actor_loss.backward()
        nn.utils.clip_grad_norm_(self.actor.parameters(), max_norm=1.0)
        self.actor_opt.step()

        # --- Temperature (alpha) update ---
        alpha_loss = -(self.log_alpha * (logp.detach() + self.target_entropy)).mean()
        self.alpha_opt.zero_grad(set_to_none=True)
        alpha_loss.backward()
        self.alpha_opt.step()

        # --- Target update ---
        soft_update_(self.q_tgt, self.q_net, self.cfg.tau)

        return {
            "critic_loss": float(critic_loss.item()),
            "actor_loss": float(actor_loss.item()),
            "alpha": float(self.alpha.item()),
        }

# ----------------------------
# Training loop
# ----------------------------
def train_sac():
    set_seed_everywhere(SEED)
    env = gym.make(ENV_ID)
    obs, _ = env.reset(seed=SEED)

    agent = SACAgent(env)
    total_steps = 0

    # Precompute action sampling helper for warm-up
    def random_action():
        return env.action_space.sample()

    ep_return, ep_len = 0.0, 0
    returns_log = []

    while total_steps < TOTAL_STEPS:
        # --- Act ---
        if total_steps < WARMUP_STEPS:
            action = random_action()
        else:
            with torch.no_grad():
                s_t = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                a_t, _ = agent.actor.sample(s_t)
                action = a_t.squeeze(0).cpu().numpy()

        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = bool(terminated or truncated)

        # Store transition (numpy for buffer)
        agent.replay.push(obs, action, next_obs, reward, float(done))

        ep_return += reward
        ep_len += 1
        total_steps += 1
        obs = next_obs

        # Learn
        if total_steps >= WARMUP_STEPS and total_steps % UPDATE_EVERY == 0:
            for _ in range(GRAD_STEPS):
                agent.update()

        # Episode end
        if done:
            returns_log.append(ep_return)
            obs, _ = env.reset()
            ep_return, ep_len = 0.0, 0

        # Logging
        if len(returns_log) and len(returns_log) % 10 == 0:
            print(f"Step {total_steps:7d} | AvgReturn(10) = {np.mean(returns_log[-10:]):.1f} | Alpha={agent.alpha.item():.3f}")

    env.close()
    print("Training finished. AvgReturn(100):", np.mean(returns_log[-100:]) if len(returns_log) >= 100 else np.mean(returns_log))

    return agent

# ----------------------------
# Greedy evaluation (mean action)
# ----------------------------
def eval_policy(agent: SACAgent, episodes: int = 5, render: bool = False) -> float:
    env = agent.env
    avg_ret = 0.0
    with torch.no_grad():
        for _ in range(episodes):
            obs, _ = env.reset(seed=np.random.randint(10_000_000))
            done = False
            ep_ret = 0.0
            while not done:
                s = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                a = agent.actor.act_deterministic(s).squeeze(0).cpu().numpy()
                obs, r, terminated, truncated, _ = env.step(a)
                ep_ret += float(r)
                done = terminated or truncated
                if render:
                    env.render()
            avg_ret += ep_ret
    return avg_ret / episodes




In [3]:
# agent = train_sac()
# avg = eval_policy(agent, episodes=10)
# print(f"Eval average return over 10 episodes: {avg:.2f}")